Copyright 2018-2026 AVEVA Group Limited

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

   http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.
SPDX-License-Identifier: Apache-2.0

In [0]:
import json
import time
import requests
from delta.tables import DeltaTable
from pyspark.sql.functions import *
from pyspark.sql.functions import to_timestamp, col

MAX_BOOKMARK_LENGTH = 4096


def validate_bookmark(value):
    if not isinstance(value, str) or not value or len(value) > MAX_BOOKMARK_LENGTH:
        raise ValueError("The API returned an invalid bookmark.")
    if any(character in value for character in "\r\n"):
        raise ValueError("The API returned an invalid bookmark.")
    return value

# Retrieve secrets from Databricks secrets
clientId = dbutils.secrets.get(scope = "cdsscope", key = "cdsclientid")
clientSecret = dbutils.secrets.get(scope = "cdsscope", key = "cdsclientsecret")

# Grab parameters from job parameters
apiVersion = dbutils.widgets.get("apiVersion")
resource = dbutils.widgets.get("resource")
tenantId = dbutils.widgets.get("tenantId")
namespaceId = dbutils.widgets.get("namespaceId")
streamIds = dbutils.widgets.get("streamIds").split(",")
catalog_name = dbutils.widgets.get("catalog_name")
schema_name = dbutils.widgets.get("schema_name")
table_name = dbutils.widgets.get("table_name")

# Use the client ID and secret to get the needed bearer token
token_endpoint = f'{resource}/identity/connect/token'
token_information = requests.post(token_endpoint,data={'client_id': clientId,'client_secret': clientSecret,'grant_type': 'client_credentials'})
token = json.loads(token_information.content)["access_token"]

In [0]:
resourceList = []
resourcesMatch = []

# Create tables needed if they don't already exist
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog_name}.{schema_name}.{table_name} (
    timestamp TIMESTAMP,
    streamId STRING,
    value DOUBLE
)
USING delta
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog_name}.{schema_name}.signupID (
    namespaceID STRING,
    signupId STRING,
    bookmark STRING
)
USING delta
""")

# Query the signupId table to check what the signupId and bookmark is for this namespace
signupId_df = spark.sql(f"""
SELECT signupId, bookmark
FROM {catalog_name}.{schema_name}.signupID 
WHERE namespaceID = '{namespaceId}'
""")
signupId_row = signupId_df.collect()
signupId = signupId_row[0]['signupId'] if signupId_row else None
bookmark = validate_bookmark(signupId_row[0]['bookmark']) if signupId_row and signupId_row[0]['bookmark'] else None

# If we already have that signupId, get its information from Cds
if signupId:
    response = requests.get(f"https://uswe.datahub.connect.aveva.com/api/{apiVersion}/Tenants/{tenantId}/Namespaces/{namespaceId}/Signups/{signupId}", headers={"Authorization": f"Bearer {token}"})
    signup = response.json()
    print("found signup id" + signupId + ". Signup is in state: " + signup['signupState'])

    response = requests.get(f"https://uswe.datahub.connect.aveva.com/api/{apiVersion}/Tenants/{tenantId}/Namespaces/{namespaceId}/Signups/{signupId}/Resources", headers={"Authorization": f"Bearer {token}"})

    resources = response.json()
    if 'resources' in resources:
        for resource in resources['resources']:
            resourceList.append(resource['resourceId'])
        resourcesMatch = set(resourceList) == set(streamIds)

# Check if the signupId is Active and if we DON'T yet have a previous bookmark
if signupId and resourcesMatch and signup['signupState'] == "Active" and not bookmark:
    # Set the bookmark
    bookmark = validate_bookmark(signup['bookmark'])
    print("Bookmark not found. Use new one.")
    print(bookmark)
    
    # Remember the bookmark for next time
    spark.sql(f"""
    MERGE INTO {catalog_name}.{schema_name}.signupID AS target
    USING (SELECT '{namespaceId}' AS namespaceID, '{signup['Id']}' AS signupId, '{bookmark}' AS bookmark) AS source
    ON target.namespaceID = source.namespaceID
    WHEN MATCHED THEN
        UPDATE SET target.signupId = source.signupId, target.bookmark = source.bookmark
    WHEN NOT MATCHED THEN
        INSERT (namespaceID, signupId, bookmark)
        VALUES (source.namespaceID, source.signupId, source.bookmark)
    """)

# Check if the signupId is active and if we have a previous bookmark
elif signupId and resourcesMatch and signup['signupState'] == "Active" and bookmark:
    #do nothing
    print( "found bookmark " + bookmark)

# If no signup already exists, create a new one
else:
    # Create signup
    signupInput = {
    "Name": "Get Updates From CONNECT Job Databricks",
    "ResourceIds": streamIds,
    "ResourceType": "Stream"
    }
    response = requests.post(f"https://uswe.datahub.connect.aveva.com/api/{apiVersion}/Tenants/{tenantId}/Namespaces/{namespaceId}/Signups/", headers={"Authorization": f"Bearer {token}"}, json=signupInput)
    createSignup = response.json()
    status = "Activating"
    while status == "Activating":
        response = requests.get(f"https://uswe.datahub.connect.aveva.com/api/{apiVersion}/Tenants/{tenantId}/Namespaces/{namespaceId}/Signups/{createSignup['id']}", headers={"Authorization": f"Bearer {token}"})
        signup = response.json()
        status = signup['signupState']
        time.sleep(1)
    bookmark = validate_bookmark(signup['bookmark'])
    signupId = signup['id']
    print("created new signup")
    print(bookmark)

    # Record signupID and bookmark. Check if a row with that namespaceId exists first and if so, update it
    spark.sql(f"""
    MERGE INTO {catalog_name}.{schema_name}.signupID AS target
    USING (SELECT '{namespaceId}' AS namespaceID, '{signupId}' AS signupId, '{bookmark}' AS bookmark) AS source
    ON target.namespaceID = source.namespaceID
    WHEN MATCHED THEN
        UPDATE SET target.signupId = source.signupId, target.bookmark = source.bookmark
    WHEN NOT MATCHED THEN
        INSERT (namespaceID, signupId, bookmark)
        VALUES (source.namespaceID, source.signupId, source.bookmark)
    """)

In [0]:
# Create empty lists to be used later
dataList = []
deleteList = []
deleteWindows = []

# Get updates using requests library. Pass the bookmark as data so it cannot alter the URL structure.
bookmark = validate_bookmark(bookmark)
response = requests.get(
    f"https://uswe.datahub.connect.aveva.com/api/{apiVersion}/Tenants/{tenantId}/Namespaces/{namespaceId}/Signups/{signupId}/Updates",
    headers={"Authorization": f"Bearer {token}"},
    params={"bookmark": bookmark}
)
updates_data = response.json()
updates = updates_data['data']
newBookmark = validate_bookmark(updates_data['bookmark'])

# Record new bookmark in table
bookmark = newBookmark
spark.sql(f"""
    MERGE INTO {catalog_name}.{schema_name}.signupID AS target
    USING (SELECT '{namespaceId}' AS namespaceID, '{signupId}' AS signupId, '{bookmark}' AS bookmark) AS source
    ON target.namespaceID = source.namespaceID
    WHEN MATCHED THEN
        UPDATE SET target.signupId = source.signupId, target.bookmark = source.bookmark
    WHEN NOT MATCHED THEN
        INSERT (namespaceID, signupId, bookmark)
        VALUES (source.namespaceID, source.signupId, source.bookmark)
    """)

# Loop through each update and check its operation type
for update in updates:
    if update['operation'] in ['Insert', 'Update', 'Replace']:
        # For each event returned, append the data to the list to be later merged into the delta table
        for event in update['events']:
            print(str(event['Timestamp']) + " " + update['resourceId'] + " " + str(event['Value']) + " " + update['operation'])
            stream = update['resourceId']
            data = [event['Timestamp'], stream, float(event['Value'])]
            dataList.append(data)
    elif update['operation'] == "Remove":
        # For each event returned, append the event to the list to be later deleted from the delta table
        for event in update['events']:
            print(str(event['Timestamp']) + " " + update['resourceId'] + " " + update['operation'])
            stream = update['resourceId']
            delete = [event['Timestamp'], stream]
            deleteList.append(delete)
    elif update['operation'] == "RemoveWindow":
        print(str(update['events'][0]['Timestamp']) + "-" + str(update['events'][1]['Timestamp']) + update['resourceId'] + " " + update['operation'])
        stream = update['resourceId']
        #record a start and end time for a delete window
        deleteWindows.append([stream, update['events'][0]['Timestamp'], update['events'][1]['Timestamp']])

# If we have data to merge, check if data exists at the same timestamp, if so update it - if not, insert
if dataList:
    df = spark.createDataFrame(dataList, ["timestamp", "streamId", "value"])
    df = df.withColumn("timestamp", to_timestamp(col("timestamp")))
    df.createOrReplaceTempView("updates_view")
    spark.sql(f"""
    MERGE INTO {catalog_name}.{schema_name}.{table_name} AS target
    USING updates_view AS source
    ON target.timestamp = source.timestamp AND target.streamId = source.streamId
    WHEN MATCHED THEN
    UPDATE SET
    target.value = source.value
    WHEN NOT MATCHED THEN
        INSERT (timestamp, streamId, value)
    VALUES (source.timestamp, source.streamId, source.value)
    """)

# If we have data to delete, delete data at those timestamps
if deleteList:
    df = spark.createDataFrame(deleteList, ["timestamp", "streamId"])
    df = df.withColumn("timestamp", to_timestamp(col("timestamp")))
    df.createOrReplaceTempView("deletes_view")
    spark.sql(f"""
    DELETE FROM {catalog_name}.{schema_name}.{table_name}
    WHERE (timestamp) IN (SELECT timestamp FROM deletes_view) AND (streamId) IN (SELECT streamId FROM deletes_view)
    """)

# If there is a delete window defined, delete data from the delta table that falls within the range
if deleteWindows:
    for window in deleteWindows:
        print(window)
        stream = window[0]
        start = window[1]
        end = window[2]
        spark.sql(f"""
        DELETE FROM {catalog_name}.{schema_name}.{table_name}
        WHERE streamId = '{stream}' AND timestamp BETWEEN to_timestamp('{start}') AND to_timestamp('{end}')
        """)